In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names
from hotelling.spatial.census import make_cell_id

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
# Load the OSM supermarkets data
osm_supermarkets = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')

In [ ]:
# Load gtfs data for public transport stops
stops = pd.read_csv(PATH_RAW / 'gtfs' / 'stops.txt')
stops = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

# Include only stops in within Berlin
# stops = stops[stops.intersects(berlin.geometry[0])]


In [ ]:
# Load the rest of the data
routes     = pd.read_csv(PATH_RAW / 'gtfs' / "routes.txt")
trips      = pd.read_csv(PATH_RAW / 'gtfs' / "trips.txt")
stop_times = pd.read_csv(PATH_RAW / 'gtfs' / "stop_times.txt")

In [ ]:
# One representative trip per route+direction (pick first)
rep_trips = (
    trips.groupby(["route_id", "direction_id"], as_index=False)
         .first()[["route_id", "direction_id", "trip_id"]]
)

# Join: route → trip → stop_times → stop coords
seq = (
    rep_trips
    .merge(stop_times[["trip_id", "stop_id", "stop_sequence"]], on="trip_id")
    .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
    .merge(routes[["route_id", "route_short_name", "route_type"]], on="route_id")
    .sort_values(["route_id", "direction_id", "stop_sequence"])
)

In [ ]:
seq.to_csv(PATH_PROCESSED / 'representative_routes.csv', index=False)

In [ ]:
seq

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    stops[stops['parent_station'].isna()].plot(ax=ax, color='royalblue', markersize=1, label='Public Transport Stops', alpha=0.5)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title('Public Transport Stops in Berlin', fontsize=16)
    plt.show()


In [ ]:
grid['cell_id'] = grid.apply(make_cell_id, axis=1)

In [ ]:
import partridge as ptg  # pip install partridge

# Bounding box: inner Ring + buffer, in WGS84
BBOX = tuple(berlin.to_crs("EPSG:4326").total_bounds)  # (min_lon, min_lat, max_lon, max_lat)

view = {
    "stops.txt": {
        "stop_lat": lambda x: (x >= BBOX[1]) & (x <= BBOX[3]),
        "stop_lon": lambda x: (x >= BBOX[0]) & (x <= BBOX[2]),
    }
}
feed = ptg.load_feed(str(PATH_RAW / "gtfs"), view=view)
# Then write trimmed feed back out
ptg.writers.write_feed_dangerously(feed, str(PATH_RAW / "gtfs_berlin"))

In [ ]:
from hotelling.spatial.distance import build_transit_travel_times

# build_transit_travel_times handles:
#   - GTFS zip construction (skipping empty files)
#   - r5py TransportNetwork creation
#   - TravelTimeMatrix computation
#   - Saving data/processed/travel_times.parquet
travel_times = build_transit_travel_times(
    grid=grid,
    supermarkets=osm_supermarkets,
    # Paths are resolved automatically from repo root.
    # Override if needed:
    # osm_pbf_path=PATH_RAW / "berlin-260512.osm.pbf",
    # gtfs_dir=PATH_RAW / "gtfs",
    # gtfs_zip=PATH_RAW / "gtfs_berlin.zip",
    # output_path=PATH_PROCESSED / "travel_times.parquet",
)

In [ ]:
# travel_times.parquet is saved automatically by build_transit_travel_times.
# Load it back for verification:
travel_times = pd.read_parquet(PATH_PROCESSED / "travel_times.parquet")

In [ ]:
travel_times = pd.read_parquet(PATH_PROCESSED / "travel_times.parquet")

In [ ]:
grid_travel_times = grid.merge(
    travel_times[["from_id", "to_id", "travel_time"]],
    left_on="cell_id",
    right_on="from_id",
    how="left"
)
grid_travel_times['if_pop'] = grid_travel_times['Einwohner'] > 0

In [ ]:
osm_supermarkets

In [ ]:
osm_supermarkets
# ID: 491
# (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())
import numpy as np
STORE_ID = np.random.choice(destinations['id'])
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=(20,18), dpi=300)
    grid_travel_times[grid_travel_times['to_id'] == STORE_ID].plot(ax=ax, column='travel_time', cmap='viridis', legend=True, alpha =0.5)
    grid_travel_times[(~grid_travel_times['if_pop']) & (grid_travel_times['to_id'] == STORE_ID) & (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())].plot(ax=ax, color='red', legend=True, alpha =0.5)
    destinations[destinations['id'] == STORE_ID].to_crs(berlin.crs).plot(ax=ax, color='yellow', markersize=5, label=f'Supermarket {STORE_ID}')
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title(f'Travel Times to Supermarket {STORE_ID}', fontsize=16)
    plt.show()

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_05_dist_matrix.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")